# Combined MR Summary Analysis

This notebook loads all MR summary CSV files from both b_scans and c_scans,
combines them into a single dataframe, and prepares the data for comprehensive analysis.

## Analysis Components:
1. Load all MR_summary CSV files from b_scans and c_scans
2. Combine into a single dataframe with temperature and scan_type columns
3. Explore the combined dataset
4. Prepare for further analysis (MR vs T, I_min/I_max vs T, etc.)

## 1. Setup & Imports

In [ ]:
# Notebook setup
from scripts.utils import setup_notebook, COLOR_B_AXIS, COLOR_C_AXIS, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Additional imports
import glob
import re

## 2. Load All MR Summary CSV Files

In [ ]:
# Define paths to MR_summary directories
b_scans_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary/b_scans"
c_scans_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary/c_scans"

# Find all CSV files
b_scan_files = list(b_scans_dir.glob("TMR_ratio_vs_V_*.csv"))
c_scan_files = list(c_scans_dir.glob("TMR_ratio_vs_V_*.csv"))

print(f"Found {len(b_scan_files)} b_scan CSV files")
print(f"Found {len(c_scan_files)} c_scan CSV files")
print(f"Total: {len(b_scan_files) + len(c_scan_files)} files")

In [ ]:
def load_mr_csv_with_metadata(csv_path, scan_type):
    """
    Load a single MR summary CSV file and add temperature and scan_type columns.
    
    Parameters:
    -----------
    csv_path : Path
        Path to the CSV file
    scan_type : str
        Either 'b_scan' or 'c_scan'
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with added 'temperature' and 'scan_type' columns
    """
    # Extract temperature from filename (e.g., TMR_ratio_vs_V_10K.csv -> 10)
    filename = csv_path.name
    temp_match = re.search(r'(\d+)K', filename)
    
    if not temp_match:
        print(f"Warning: Could not extract temperature from {filename}")
        return None
    
    temperature = int(temp_match.group(1))
    
    # Load CSV
    df = pd.read_csv(csv_path)
    
    # Add metadata columns
    df['temperature'] = temperature
    df['scan_type'] = scan_type
    
    return df

# Load all b_scan files
b_scan_dataframes = []
for csv_file in b_scan_files:
    df = load_mr_csv_with_metadata(csv_file, 'b_scan')
    if df is not None:
        b_scan_dataframes.append(df)

# Load all c_scan files
c_scan_dataframes = []
for csv_file in c_scan_files:
    df = load_mr_csv_with_metadata(csv_file, 'c_scan')
    if df is not None:
        c_scan_dataframes.append(df)

print(f"\nSuccessfully loaded:")
print(f"  b_scans: {len(b_scan_dataframes)} dataframes")
print(f"  c_scans: {len(c_scan_dataframes)} dataframes")

## 3. Combine All Data into Single DataFrame

In [ ]:
# Combine all dataframes
all_dataframes = b_scan_dataframes + c_scan_dataframes
df_combined = pd.concat(all_dataframes, ignore_index=True)

print(f"\n{'='*70}")
print(f"Combined DataFrame Created!")
print(f"{'='*70}")
print(f"Total rows: {len(df_combined):,}")
print(f"Total columns: {len(df_combined.columns)}")
print(f"\nColumns: {list(df_combined.columns)}")
print(f"\nTemperature range: {df_combined['temperature'].min()}K to {df_combined['temperature'].max()}K")
print(f"Unique temperatures: {sorted(df_combined['temperature'].unique())}")
print(f"\nScan types: {df_combined['scan_type'].unique()}")
print(f"\nData shape: {df_combined.shape}")

In [ ]:
# Display first few rows
print("\nFirst 10 rows of combined data:")
df_combined.head(10)

In [ ]:
# Display summary statistics
print("\nSummary Statistics:")
df_combined.describe()

## 4. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df_combined.isnull().sum())
print(f"\nTotal missing values: {df_combined.isnull().sum().sum()}")
print(f"Percentage of missing data: {100 * df_combined.isnull().sum().sum() / (df_combined.shape[0] * df_combined.shape[1]):.2f}%")

In [ ]:
# Count data points per temperature and scan type
print("\nData points per temperature and scan type:")
pivot_counts = df_combined.groupby(['temperature', 'scan_type']).size().unstack(fill_value=0)
print(pivot_counts)

## 5. Save Combined DataFrame

In [ ]:
# Save combined dataframe for future use
output_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary"
output_dir.mkdir(parents=True, exist_ok=True)

# Save as pickle for fast loading with data types preserved
pkl_path = output_dir / "MR_summary_combined_all_temps.pkl"
df_combined.to_pickle(pkl_path)
print(f"Saved combined dataframe (pickle): {pkl_path}")

# Save as CSV for portability
csv_path = output_dir / "MR_summary_combined_all_temps.csv"
df_combined.to_csv(csv_path, index=False)
print(f"Saved combined dataframe (CSV): {csv_path}")

## 6. Quick Visualization: MR vs Voltage for All Temperatures

In [ ]:
# ============================================================
# FILTER CONFIGURATION - Adjust these values as needed
# ============================================================

# Minimum I_apar threshold (A) - removes noisy low-current data
i_apar_threshold = 2e-9  # Remove data where |I_apar| < 1 nA

# Voltage cutoffs per temperature (K: max_voltage in V)
# Only temperatures listed here will have voltage limits applied
# Leave empty {} to show full voltage range for all temperatures


# Plot style configuration
plot_with_errorbars = True  # Set to False to plot without error bars (faster)
errorbar_alpha = 0.6  # Transparency of error bars (0-1)

# ============================================================


def apply_filters(data, temp, i_threshold, v_cutoffs):
    """
    Apply current and voltage filters to TMR data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame subset for a specific temperature/scan_type
    temp : int or float
        Temperature value
    i_threshold : float
        Minimum |I_apar| threshold in Amperes
    v_cutoffs : dict
        Dictionary of {temperature: max_voltage} for voltage cutoffs
    
    Returns:
    --------
    pd.DataFrame
        Filtered DataFrame
    """
    # Filter 1: Remove data where |I_apar| < threshold
    filtered = data[np.abs(data['I_apar (A)']) >= i_threshold].copy()
    
    # Filter 2: Apply voltage cutoff if specified for this temperature
    if temp in v_cutoffs:
        max_v = v_cutoffs[temp]
        filtered = filtered[np.abs(filtered['Voltage (V)']) <= max_v]
    
    return filtered
v_cutoffs = {}

temperatures = [20,30,40,50,60,70]

# ── Figure 1: b_scans ────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs={})
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax1.errorbar(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax1.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')

ax1.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax1.set_ylabel('MR ratio (%)')
#ax1.grid(False)
ax1.set_ylim(0, 400)
ax1.legend(loc='upper left', borderaxespad=0, frameon=True)

fig1.tight_layout()
plt.show()

# ── Figure 2: c_scans ────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs={})
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax2.errorbar( data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax2.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')
ax2.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax2.set_ylabel('MR ratio (%)')
#ax2.grid(False)
ax2.set_ylim(0, 400 )
ax2.legend(loc='upper left', borderaxespad=0, frameon=True)

fig2.tight_layout()
plt.show()


In [ ]:
# ============================================================
# FILTER CONFIGURATION - Adjust these values as needed
# ============================================================

# Minimum I_apar threshold (A) - removes noisy low-current data
i_apar_threshold = 2e-9  # Remove data where |I_apar| < 1 nA

# Voltage cutoffs per temperature (K: max_voltage in V)
# Only temperatures listed here will have voltage limits applied
# Leave empty {} to show full voltage range for all temperatures


# Plot style configuration
plot_with_errorbars = True  # Set to False to plot without error bars (faster)
errorbar_alpha = 0.6  # Transparency of error bars (0-1)

# ============================================================


def apply_filters(data, temp, i_threshold, v_cutoffs):
    """
    Apply current and voltage filters to TMR data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame subset for a specific temperature/scan_type
    temp : int or float
        Temperature value
    i_threshold : float
        Minimum |I_apar| threshold in Amperes
    v_cutoffs : dict
        Dictionary of {temperature: max_voltage} for voltage cutoffs
    
    Returns:
    --------
    pd.DataFrame
        Filtered DataFrame
    """
    # Filter 1: Remove data where |I_apar| < threshold
    filtered = data[np.abs(data['I_apar (A)']) >= i_threshold].copy()
    
    # Filter 2: Apply voltage cutoff if specified for this temperature
    if temp in v_cutoffs:
        max_v = v_cutoffs[temp]
        filtered = filtered[np.abs(filtered['Voltage (V)']) <= max_v]
    
    return filtered


temperatures = [80, 90, 100, 110, 120, 130, 140, 150, 160]

# ── Figure 1: b_scans ────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs)
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax1.errorbar(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax1.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')

ax1.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax1.set_ylabel('MR ratio (%)')
#ax1.grid(False)
ax1.set_ylim(0, 400)
ax1.legend(loc='upper left', borderaxespad=0, frameon=True, ncol=2)

fig1.tight_layout()
plt.show()

# ── Figure 2: c_scans ────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs)
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax2.errorbar( data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax2.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')

ax2.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax2.set_ylabel('MR ratio (%)')
#ax2.grid(False)
ax2.set_ylim(0, 400)
ax2.legend(loc='upper left', borderaxespad=0, frameon=True, ncol=2)

fig2.tight_layout()
plt.show()


In [ ]:
# Calculate and plot both positive and negative side voltage peaks at peak TMR vs temperature
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
voltage_cutoffs = {}

peak_data_pos = []
peak_data_neg = []


if 'df_combined' in locals() or 'df_combined' in globals():
    temperatures = df_combined['temperature'].unique()
    temperatures.sort()

    for i, temp in enumerate(temperatures):
        for scan_type in ['b_scan', 'c_scan']:
            data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == scan_type)]
            if len(data) == 0:
                continue
            
            try:
                filtered_data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
            except NameError:
                filtered_data = data
                
            # Split data into positive and negative voltage regions
            data_pos = filtered_data[filtered_data['Voltage (V)'] > 0]
            data_neg = filtered_data[filtered_data['Voltage (V)'] < 0]
            
            # Positive side peak
            if len(data_pos) > 0:
                max_tmr_idx_pos = data_pos['TMR_Ratio'].idxmax()
                peak_voltage_pos = data_pos.loc[max_tmr_idx_pos, 'Voltage (V)']
                max_tmr_pos = data_pos.loc[max_tmr_idx_pos, 'TMR_Ratio']
                peak_data_pos.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak_voltage_pos,
                    'max_tmr': max_tmr_pos
                })
                
            # Negative side peak
            if len(data_neg) > 0:
                max_tmr_idx_neg = data_neg['TMR_Ratio'].idxmax()
                peak_voltage_neg = data_neg.loc[max_tmr_idx_neg, 'Voltage (V)']
                max_tmr_neg = data_neg.loc[max_tmr_idx_neg, 'TMR_Ratio']
                peak_data_neg.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak_voltage_neg,
                    'max_tmr': max_tmr_neg
                })

    # Create the plot
    fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

    if peak_data_pos:
        df_pos = pd.DataFrame(peak_data_pos)
        b_scan_pos = df_pos[df_pos['scan_type'] == 'b_scan']
        c_scan_pos = df_pos[df_pos['scan_type'] == 'c_scan']
        
        if not b_scan_pos.empty:
            ax.plot(b_scan_pos['temperature'], b_scan_pos['peak_voltage'], 'o-', label='b scan ($V$ > 0)', color=COLOR_B_AXIS, linewidth=2, markersize=8)
        if not c_scan_pos.empty:
            ax.plot(c_scan_pos['temperature'], c_scan_pos['peak_voltage'], 's-', label='c scan ($V$ > 0)', color=COLOR_C_AXIS, linewidth=2, markersize=8)

    if peak_data_neg:
        df_neg = pd.DataFrame(peak_data_neg)
        b_scan_neg = df_neg[df_neg['scan_type'] == 'b_scan']
        c_scan_neg = df_neg[df_neg['scan_type'] == 'c_scan']
        
        if not b_scan_neg.empty:
            ax.plot(b_scan_neg['temperature'], b_scan_neg['peak_voltage'], 'v--', label='b scan ($V$ < 0)', color='#56B4E9', linewidth=2, markersize=8)
        if not c_scan_neg.empty:
            ax.plot(c_scan_neg['temperature'], c_scan_neg['peak_voltage'], 'v--', label='c scan ($V$ < 0)', color='#E69F00', linewidth=2, markersize=8)

    ax.set_xlabel('$T$ (K)')
    ax.set_ylabel('$V_{bias}^{max(MR)}$')
    #ax.set_title('Positive and Negative Side Voltage Peaks vs Temperature', fontweight='bold')
    
    # Place legend outside to avoid obscuring data
    ax.legend(loc='best', borderaxespad=0, frameon=False)
    
    plt.xlim(15,165)
    plt.ylim(-0.6,0.6)
    plt.tight_layout()
    plt.show()
else:
    print("df_combined not found in namespace.")


In [ ]:
import numpy as np

def plot_fowler_nordheim(ax, scan_type, I_col, title, temperatures):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type) &
            (df_combined['Voltage (V)'].abs() >= 0.1)
        ].copy()
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            x = 1 / data['Voltage (V)']
            y = np.log(data[I_col].abs() / data['Voltage (V)']**2)
            ax.plot(x, y, 'o', color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')
    ax.set_xlabel('$1/V_{\mathrm{bias}}$ (V$^{-1}$)')
    ax.set_ylabel('$\ln(|I|/V_{\mathrm{bias}}^2)$')
    ax.set_title(title, fontweight='bold')
    ax.grid(False)
    ax.legend()

temperatures = [100, 90, 80, 70, 60, 50, 30, 20]

plots = [
    ('b_scan', 'I_par (A)',  'Fowler-Nordheim Plot (FM) - b_scans'),
    ('c_scan', 'I_par (A)',  'Fowler-Nordheim Plot (FM) - c_scans'),
    ('b_scan', 'I_apar (A)', 'Fowler-Nordheim Plot (AFM) - b_scans'),
    ('c_scan', 'I_apar (A)', 'Fowler-Nordheim Plot (AFM) - c_scans'),
]

for scan_type, I_col, title in plots:
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_fowler_nordheim(ax, scan_type, I_col, title, temperatures)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np

trim = 3

def plot_dIdV(ax, scan_type, I_col, title, temperatures):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            voltage = data['Voltage (V)'].values
            dIdV = np.gradient(data[I_col].values * 1e6, voltage)
            ax.plot(voltage[trim:-trim], dIdV[trim:-trim], 'o', color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')
    ax.set_xlabel('$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$dI/dV$ (μA/V)')
    ax.set_title(title, fontweight='bold')
    ax.grid(False)
    ax.legend(ncol=2)

temps_afm = [20, 30, 40, 50, 60, 70, 80, 90, 100]
temps_fm  = [20, 30, 50, 60, 70, 80, 90, 100]

plots = [
    ('b_scan', 'I_apar (A)', 'Raw dI/dV (AFM) - b_scans', temps_afm),
    ('c_scan', 'I_apar (A)', 'Raw dI/dV (AFM) - c_scans', temps_afm),
    ('b_scan', 'I_par (A)',  'Raw dI/dV (FM) - b_scans',  temps_fm),
    ('c_scan', 'I_par (A)',  'Raw dI/dV (FM) - c_scans',  temps_fm),
]

for scan_type, I_col, title, temperatures in plots:
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_dIdV(ax, scan_type, I_col, title, temperatures)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd

trim = 3

# Convert peak_data to DataFrames for easy lookup
df_pos = pd.DataFrame(peak_data_pos) if peak_data_pos else pd.DataFrame()

def compute_normalized_dIdV(data, I_col):
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    dIdV = np.gradient(I, V)
    V, I, dIdV = V[trim:-trim], I[trim:-trim], dIdV[trim:-trim]
    mask = np.abs(V) > 0.05
    return V[mask], dIdV[mask] / (I[mask] / V[mask])

def get_normalized_didv_at_voltage(data, I_col, peak_voltage):
    V_masked, normalized = compute_normalized_dIdV(data, I_col)
    idx = np.argmin(np.abs(V_masked - peak_voltage))
    return V_masked[idx], normalized[idx]

def add_peak_markers(ax, scan_type, I_col, df_pos, temperatures_set):
    for _, row in df_pos[df_pos['scan_type'] == scan_type].iterrows():
        temp = row['temperature']
        if temp not in temperatures_set:
            continue
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy()
        if len(data) == 0:
            continue
        i = temperatures.index(temp)
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        V_pt, y_pt = get_normalized_didv_at_voltage(data, I_col, row['peak_voltage'])
        ax.scatter(V_pt, y_pt, marker='o', s=100, facecolors=color,
                   edgecolors='#000000', linewidths=1.2, zorder=6)

temperatures = [20, 30, 40, 50, 60, 70, 80, 90, 100]
temperatures_set = set(temperatures)

plots = [
    ('b_scan', 'I_apar (A)', 'Normalized dI/dV (AFM) — b-axis', False),
    ('c_scan', 'I_apar (A)', 'Normalized dI/dV (AFM) — c-axis', False),
    ('b_scan', 'I_par (A)',  'Normalized dI/dV (FM) — b-axis',  True),
    ('c_scan', 'I_par (A)',  'Normalized dI/dV (FM) — c-axis',  True),
]

for scan_type, I_col, title, add_markers in plots:
    fig, ax = plt.subplots(figsize=(6, 5))

    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            V_masked, normalized = compute_normalized_dIdV(data, I_col)
            ax.plot(V_masked, normalized, 'o', color=color, alpha=0.7, label=f'{temp} K')

    if add_markers:
        add_peak_markers(ax, scan_type, I_col, df_pos, temperatures_set)

    ax.set_xlabel('$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    ax.legend(loc='upper left', borderaxespad=0, ncol=2, frameon=True)
    fig.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

TEMP = 20
trim = 2

def compute_normalized_didv(data, I_col):
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    dIdV = np.gradient(I, V)
    V, I, dIdV = V[trim:-trim], I[trim:-trim], dIdV[trim:-trim]
    mask = np.abs(V) > 0.05
    return V[mask], dIdV[mask] / (I[mask] / V[mask])


def plot_axis(ax, scan_type, title):
    data = df_combined[
        (df_combined['temperature'] == TEMP) &
        (df_combined['scan_type'] == scan_type)
    ].copy()

    if len(data) == 0:
        ax.set_title(f'{title} — no data at {TEMP} K')
        return

    V_afm, norm_afm = compute_normalized_didv(data, 'I_apar (A)')
    ax.plot(V_afm, norm_afm, 'o', color=OKABE_ITO_CYCLE[0], alpha=0.85,
            markersize=4, linewidth=1.4, label='AFM')

    V_fm, norm_fm = compute_normalized_didv(data, 'I_par (A)')
    ax.plot(V_fm, norm_fm, 'o', color=OKABE_ITO_CYCLE[1], alpha=0.85,
            markersize=4, linewidth=1.4, label='FM')

    peaks = {}
    for norm, V, label in [(norm_afm, V_afm, 'AFM'), (norm_fm, V_fm, 'FM')]:
        mask = (V >= 0.5) & (V <= 0.9)
        if mask.any():
            peak_V = V[mask][np.argmax(norm[mask])]
            peaks[label] = peak_V
            ax.axvline(peak_V, color='black', linewidth=2.0,
                       linestyle='--', alpha=0.9, zorder=5)

    if 'AFM' in peaks and 'FM' in peaks:
        V_afm_peak = peaks['AFM']
        V_fm_peak  = peaks['FM']
        delta_V    = abs(V_fm_peak - V_afm_peak)
        arrow_y    = 1.5
        mid_V      = (V_afm_peak + V_fm_peak) / 2

        ax.annotate('', xy=(V_fm_peak, arrow_y), xytext=(V_afm_peak, arrow_y),
                    arrowprops=dict(arrowstyle='<->', color='black',
                                   lw=1.5, mutation_scale=14))
        ax.text(mid_V, arrow_y + 1.15,
                f'$\\Delta V = {delta_V*1000:.0f}$ mV',
                ha='center', va='top', color='black', fontsize=14,
                rotation=90)

    ax.set_xlabel('$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_xlim(-1, 1)
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    ax.legend()


for scan_type, title in [('b_scan', 'b-axis'), ('c_scan', 'c-axis')]:
    fig, ax = plt.subplots(figsize=(6, 5))
    plot_axis(ax, scan_type=scan_type, title=title)
    fig.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration ---
temperatures = [20, 30, 40, 50, 60, 70, 80, 90, 100]

# --- Figure 1: d²I/dV² — AFM state ---
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for scan_type, ax in zip(['b_scan', 'c_scan'], [ax1, ax2]):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) & 
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')
        
        if len(data) > 2:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            V = data['Voltage (V)'].values
            I = data['I_apar (A)'].values
            
            # --- Calculation: d²I/dV² ---
            dIdV = np.gradient(I, V)
            d2IdV2 = np.gradient(dIdV, V)
            
            # --- Updated Masking Mechanism (0.05 < |V| < 0.9) ---
            mask = (np.abs(V) > 0.01) & (np.abs(V) < 0.9)
            
            ax.plot(V[mask], d2IdV2[mask], 'o', color=color, alpha=0.7, markersize=3, label=f'{temp}K')

    ax.set_xlabel('$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(d^2I/dV^2)$ (A/V$^2$)')
    ax.set_title(f'$d^2I/dV^2$ (AFM) - {scan_type}', fontweight='bold')
    ax.grid(False)
    ax.legend(ncol=2)
    ax.set_ylim(-1E-5, 1E-5)

fig1.tight_layout()
plt.show()

# --- Figure 2: d²I/dV² — FM state ---
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(16, 6))

for scan_type, ax in zip(['b_scan', 'c_scan'], [ax3, ax4]):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) & 
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')
        
        if len(data) > 2:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            V = data['Voltage (V)'].values
            I = data['I_par (A)'].values 
            
            # --- Calculation: d²I/dV² ---
            dIdV = np.gradient(I, V)
            d2IdV2 = np.gradient(dIdV, V)
            
            # --- Updated Masking Mechanism (0.05 < |V| < 0.9) ---
            mask = (np.abs(V) > 0.01) & (np.abs(V) < 0.9)
            
            ax.plot(V[mask], d2IdV2[mask], 'o', color=color, alpha=0.7, markersize=3, label=f'{temp}K')

    ax.set_xlabel('$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(d^2I/dV^2)$ (A/V$^2$)')
    ax.set_title(f'$d^2I/dV^2$ (FM) - {scan_type}', fontweight='bold')
    ax.grid(False)
    ax.legend(ncol=2)
    ax.set_ylim(-1E-5, 1E-5)

fig2.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Convert peak_data to DataFrames for easy lookup
df_pos = pd.DataFrame(peak_data_pos) if peak_data_pos else pd.DataFrame()
df_neg = pd.DataFrame(peak_data_neg) if peak_data_neg else pd.DataFrame()

TEMP = 50  # K — single temperature of interest

def compute_normalized_didv(data, I_col):
    """Return (V_masked, normalized dI/dV) for a single temperature/scan dataset."""
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    dIdV = np.gradient(I, V)
    IoverV = I / V
    mask = np.abs(V) > 0.05
    return V[mask], dIdV[mask] / IoverV[mask]


def get_normalized_didv_at_voltage(data, I_col, peak_voltage):
    """Return (V, normalized dI/dV) at the point closest to peak_voltage."""
    V_masked, normalized = compute_normalized_didv(data, I_col)
    idx = np.argmin(np.abs(V_masked - peak_voltage))
    return V_masked[idx], normalized[idx]


def add_peak_markers(ax, scan_type, I_col, df_pos, df_neg):
    """Overlay star markers at peak TMR voltages on a normalized dI/dV axis."""
    for df in [df_pos, df_neg]:
        if df.empty:
            continue
        subset = df[(df['scan_type'] == scan_type) & (df['temperature'] == TEMP)]
        for _, row in subset.iterrows():
            data = df_combined[
                (df_combined['temperature'] == TEMP) &
                (df_combined['scan_type'] == scan_type)
            ].copy()
            if len(data) == 0:
                continue
            V_pt, y_pt = get_normalized_didv_at_voltage(data, I_col, row['peak_voltage'])
            ax.scatter(V_pt, y_pt, marker='*', s=180, facecolors='white',
                       edgecolors='#D55E00', linewidths=1.2, zorder=6,
                       label='Peak MR' if _ == subset.index[0] else '')


def plot_axis(ax, scan_type, title):
    """
    Plot AFM (I_apar) and FM (I_par) normalized dI/dV at TEMP K on a single axes.

    Visual encoding
    ---------------
    State   Color       Line style
    AFM     #0072B2     dashed  (--)
    FM      #009E73     solid   (-)
    """
    data = df_combined[
        (df_combined['temperature'] == TEMP) &
        (df_combined['scan_type'] == scan_type)
    ].copy()

    if len(data) == 0:
        ax.set_title(f'{title} — no data at {TEMP} K')
        return

    # --- AFM ---
    V_afm, norm_afm = compute_normalized_didv(data, 'I_apar (A)')
    ax.plot(V_afm, norm_afm, 'o', color='#0072B2', alpha=0.85,
            markersize=4, linewidth=1.4, label='AFM')

    # --- FM ---
    V_fm, norm_fm = compute_normalized_didv(data, 'I_par (A)')
    ax.plot(V_fm, norm_fm, 'o', color='#009E73', alpha=0.85,
            markersize=4, linewidth=1.4, label='FM')

    # --- Peak markers on FM ---
    add_peak_markers(ax, scan_type, 'I_par (A)', df_pos, df_neg)

    ax.set_xlabel('$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    ax.legend()


# --- Figure 1: b-axis at 20 K ---
fig1, ax1 = plt.subplots(figsize=(7, 5))
plot_axis(ax1, scan_type='b_scan', title=f'Normalized dI/dV — b-axis, {TEMP} K')
fig1.tight_layout()
plt.show()

# --- Figure 2: c-axis at 20 K ---
fig2, ax2 = plt.subplots(figsize=(7, 5))
plot_axis(ax2, scan_type='c_scan', title=f'Normalized dI/dV — c-axis, {TEMP} K')
fig2.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Convert peak_data to DataFrames for easy lookup
df_pos = pd.DataFrame(peak_data_pos) if peak_data_pos else pd.DataFrame()
df_neg = pd.DataFrame(peak_data_neg) if peak_data_neg else pd.DataFrame()

TEMP = 70  # K — single temperature of interest

def compute_normalized_didv(data, I_col):
    """Return (V_masked, normalized dI/dV) for a single temperature/scan dataset."""
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    dIdV = np.gradient(I, V)
    IoverV = I / V
    mask = np.abs(V) > 0.05
    return V[mask], dIdV[mask] / IoverV[mask]


def get_normalized_didv_at_voltage(data, I_col, peak_voltage):
    """Return (V, normalized dI/dV) at the point closest to peak_voltage."""
    V_masked, normalized = compute_normalized_didv(data, I_col)
    idx = np.argmin(np.abs(V_masked - peak_voltage))
    return V_masked[idx], normalized[idx]


def add_peak_markers(ax, scan_type, I_col, df_pos, df_neg):
    """Overlay star markers at peak TMR voltages on a normalized dI/dV axis."""
    for df in [df_pos, df_neg]:
        if df.empty:
            continue
        subset = df[(df['scan_type'] == scan_type) & (df['temperature'] == TEMP)]
        for _, row in subset.iterrows():
            data = df_combined[
                (df_combined['temperature'] == TEMP) &
                (df_combined['scan_type'] == scan_type)
            ].copy()
            if len(data) == 0:
                continue
            V_pt, y_pt = get_normalized_didv_at_voltage(data, I_col, row['peak_voltage'])
            ax.scatter(V_pt, y_pt, marker='*', s=180, facecolors='white',
                       edgecolors='#D55E00', linewidths=1.2, zorder=6,
                       label='Peak MR' if _ == subset.index[0] else '')


def plot_axis(ax, scan_type, title):
    """
    Plot AFM (I_apar) and FM (I_par) normalized dI/dV at TEMP K on a single axes.

    Visual encoding
    ---------------
    State   Color       Line style
    AFM     #0072B2     dashed  (--)
    FM      #009E73     solid   (-)
    """
    data = df_combined[
        (df_combined['temperature'] == TEMP) &
        (df_combined['scan_type'] == scan_type)
    ].copy()

    if len(data) == 0:
        ax.set_title(f'{title} — no data at {TEMP} K')
        return

    # --- AFM ---
    V_afm, norm_afm = compute_normalized_didv(data, 'I_apar (A)')
    ax.plot(V_afm, norm_afm, 'o', color='#0072B2', alpha=0.85,
            markersize=4, linewidth=1.4, label='AFM')

    # --- FM ---
    V_fm, norm_fm = compute_normalized_didv(data, 'I_par (A)')
    ax.plot(V_fm, norm_fm, 'o', color='#009E73', alpha=0.85,
            markersize=4, linewidth=1.4, label='FM')

    # --- Peak markers on FM ---
    add_peak_markers(ax, scan_type, 'I_par (A)', df_pos, df_neg)

    ax.set_xlabel('$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    ax.legend()


# --- Figure 1: b-axis at 20 K ---
fig1, ax1 = plt.subplots(figsize=(7, 5))
plot_axis(ax1, scan_type='b_scan', title=f'Normalized dI/dV — b-axis, {TEMP} K')
fig1.tight_layout()
plt.show()

# --- Figure 2: c-axis at 20 K ---
fig2, ax2 = plt.subplots(figsize=(7, 5))
plot_axis(ax2, scan_type='c_scan', title=f'Normalized dI/dV — c-axis, {TEMP} K')
fig2.tight_layout()
plt.show()

## 9. I_par at Fixed Voltage vs Temperature

Analyze how the parallel current (I_par) varies with temperature at a fixed bias voltage of 0.5V.

In [ ]:
# Extract I_par at V=0.25V for each temperature and scan type
target_voltage = 0.1  # V
tolerance = 0.01  # Voltage tolerance for matching

def get_current_at_voltage(group, v_target, tolerance=0.01):
    """Extract current closest to target voltage."""
    voltage_diff = np.abs(group['Voltage (V)'] - v_target)
    closest_idx = voltage_diff.idxmin()
    
    if voltage_diff.loc[closest_idx] < tolerance:
        return pd.Series({
            'I_par': group.loc[closest_idx, 'I_par (A)'],
            'I_par_error': group.loc[closest_idx, 'I_par_error (A)'],
            'voltage_actual': group.loc[closest_idx, 'Voltage (V)'],
            'I_apar': group.loc[closest_idx, 'I_apar (A)'],
            'I_apar_error': group.loc[closest_idx, 'I_apar_error (A)']
        })
    else:
        return pd.Series({
            'I_par': np.nan,
            'I_par_error': np.nan,
            'voltage_actual': np.nan,
            'I_apar': np.nan,
            'I_apar_error': np.nan
        })

# Apply to each temperature/scan_type group
i_par_vs_temp = df_combined.groupby(['temperature', 'scan_type']).apply(
    lambda g: get_current_at_voltage(g, target_voltage, tolerance)
).reset_index()

# Separate b_scan and c_scan
b_scan_data = i_par_vs_temp[i_par_vs_temp['scan_type'] == 'b_scan'].sort_values('temperature')
c_scan_data = i_par_vs_temp[i_par_vs_temp['scan_type'] == 'c_scan'].sort_values('temperature')

print(f"I_par at {target_voltage}V vs Temperature")
print(f"\nb_scans:")
print(b_scan_data)
print(f"\nc_scans:")
print(c_scan_data)

In [ ]:
# Plot I_par at 0.5V vs Temperature for b and c-axis
fig, ax = plt.subplots(figsize=(6, 5), dpi = 600)

# Plot b_scans
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'],
            yerr=b_scan_data['I_par_error'],
            fmt='o-', color=OKABE_ITO_CYCLE[0], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis, FM', alpha=0.8)

# Plot c_scans
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'],
            yerr=c_scan_data['I_par_error'],
            fmt='s-', color=OKABE_ITO_CYCLE[1], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis,FM', alpha=0.8)
# Plot b_scans
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'],
            yerr=b_scan_data['I_apar_error'],
            fmt='o-', color=OKABE_ITO_CYCLE[2], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='AFM', alpha=0.8)


ax.set_yscale('log')

ax.set_xlabel('$T_K$')
ax.set_ylabel('$I$ (A)')
#ax.set_title(f'Parallel Current (I_par) at {target_voltage}V vs Temperature', fontweight='bold')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best')
ax.set_xlim(20,160)
ax.set_ylim(5E-10,4E-7)

plt.tight_layout()
plt.show()

In [ ]:
# Plot I_par at 0.5V vs Temperature for b and c-axis
fig, ax = plt.subplots(figsize=(6, 5), dpi = 600)

# Plot b_scans
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'],
            yerr=b_scan_data['I_par_error'],
            fmt='o-', color=OKABE_ITO_CYCLE[0], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis, FM', alpha=0.8)

# Plot c_scans
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'],
            yerr=c_scan_data['I_par_error'],
            fmt='s-', color=OKABE_ITO_CYCLE[1], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis,FM', alpha=0.8)



ax.set_yscale('log')

ax.set_xlabel('$T_K$')
ax.set_ylabel('$I$ (A)')
#ax.set_title(f'Parallel Current (I_par) at {target_voltage}V vs Temperature', fontweight='bold')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best')
ax.set_xlim(20,160)
ax.set_ylim(5E-10,4E-7)

plt.tight_layout()
plt.show()

In [ ]:
# Plot I_apar and I_par at 0.25V vs Temperature with dual y-axes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left subplot: b-axis
ax1_right = ax1.twinx()  # Create right y-axis for I_par

# Plot I_apar on left axis (convert to µA)
line1 = ax1.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'] * 1e6,
                     yerr=b_scan_data['I_apar_error'] * 1e6,
                     fmt='o-', color=COLOR_B_AXIS, markersize=8, linewidth=2,
                     capsize=5, capthick=2, label='I_apar', alpha=0.8)

# Plot I_par on right axis (keep in A, convert to µA for display)
line2 = ax1_right.errorbar(b_scan_data['temperature'], b_scan_data['I_par'] * 1e6,
                            yerr=b_scan_data['I_par_error'] * 1e6,
                            fmt='s-', color='#56B4E9', markersize=8, linewidth=2,
                            capsize=5, capthick=2, label='I_par', alpha=0.8)

ax1.set_xlabel('$T_K$')
ax1.set_ylabel('$I_{AFM}$ (µA)', color=COLOR_B_AXIS)
ax1_right.set_ylabel('$I_{FM}$ (µA)', color='#56B4E9')
ax1.set_title(f'b-axis: I_apar and I_par at {target_voltage}V vs Temperature', fontweight='bold')
ax1.tick_params(axis='y', labelcolor=COLOR_B_AXIS)
ax1_right.tick_params(axis='y', labelcolor='#56B4E9')
ax1.grid(False, alpha=0.3, linestyle='--')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_right.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Right subplot: c-axis
ax2_right = ax2.twinx()  # Create right y-axis for I_par

# Plot I_apar on left axis (convert to µA)
line3 = ax2.errorbar(c_scan_data['temperature'], c_scan_data['I_apar'] * 1e6,
                     yerr=c_scan_data['I_apar_error'] * 1e6,
                     fmt='o-', color=COLOR_C_AXIS, markersize=8, linewidth=2,
                     capsize=5, capthick=2, label='I_apar', alpha=0.8)

# Plot I_par on right axis (convert to µA for display)
line4 = ax2_right.errorbar(c_scan_data['temperature'], c_scan_data['I_par'] * 1e6,
                            yerr=c_scan_data['I_par_error'] * 1e6,
                            fmt='s-', color='#E69F00', markersize=8, linewidth=2,
                            capsize=5, capthick=2, label='I_par', alpha=0.8)

ax2.set_xlabel('$T_K$')
ax2.set_ylabel('$I_{AFM}$ (µA)', color=COLOR_C_AXIS)
ax2_right.set_ylabel('$I_{FM}$ (µA)', color='#E69F00')
ax2.set_title(f'c-axis: I_apar and I_par at {target_voltage}V vs Temperature', fontweight='bold')
ax2.tick_params(axis='y', labelcolor=COLOR_C_AXIS)
ax2_right.tick_params(axis='y', labelcolor='#E69F00')
ax2.grid(False, alpha=0.3, linestyle='--')

# Combine legends
lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2_right.get_legend_handles_labels()
ax2.legend(lines3 + lines4, labels3 + labels4, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Compare I_par at 0.25V: b-axis vs c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Plot b-axis I_par
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'] * 1e6,
            yerr=b_scan_data['I_par_error'] * 1e6,
            fmt='o-', color=COLOR_B_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c-axis I_par
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'] * 1e6,
            yerr=c_scan_data['I_par_error'] * 1e6,
            fmt='s-', color=COLOR_C_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('$T_K$')
ax.set_ylabel('$I_{FM}$ (µA)')
ax.set_title(f'Parallel Current (I_par) at {target_voltage}V vs Temperature: b-axis vs c-axis', fontweight='bold')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best') # Set lower limit for log scale
plt.tight_layout()
plt.show()

In [ ]:
# Compare I_apar at 0.25V: b-axis vs c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Plot b-axis I_apar
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'] * 1e6,
            yerr=b_scan_data['I_apar_error'] * 1e6,
            fmt='o-', color=COLOR_B_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c-axis I_apar
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_apar'] * 1e6,
            yerr=c_scan_data['I_apar_error'] * 1e6,
            fmt='s-', color=COLOR_C_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('$T_K$')
ax.set_ylabel('$I_{AFM}$ (µA)')
ax.set_title(f'Anti-parallel Current (I_apar) at {target_voltage}V vs Temperature: b-axis vs c-axis', fontweight='bold')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate and plot voltage at peak TMR vs temperature
import pandas as pd
import matplotlib.pyplot as plt

peak_tmr_data = []

# Assuming df_combined and apply_filters are in the namespace
if 'df_combined' in locals() or 'df_combined' in globals():
    temperatures = df_combined['temperature'].unique()
    temperatures.sort()

    for i, temp in enumerate(temperatures):
        for scan_type in ['b_scan', 'c_scan']:
            data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == scan_type)]
            if len(data) == 0:
                continue
            
            # Use 'i_apar_threshold' and 'voltage_cutoffs' if defined
            try:
                filtered_data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
            except NameError:
                # Fallback if filters are not available in the current namespace
                filtered_data = data
                
            if len(filtered_data) > 0:
                # Find the row with maximum TMR ratio (%)
                max_tmr_idx = filtered_data['TMR_Ratio'].idxmax()
                peak_voltage = filtered_data.loc[max_tmr_idx, 'Voltage (V)']
                max_tmr = filtered_data.loc[max_tmr_idx, 'TMR_Ratio']
                peak_tmr_data.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak_voltage,
                    'max_tmr': max_tmr
                })

    if peak_tmr_data:
        df_peak_tmr = pd.DataFrame(peak_tmr_data)

        # Create the plot
        fig, ax = plt.subplots(figsize=(10, 6))

        # Plot b_scan
        b_scan_peaks = df_peak_tmr[df_peak_tmr['scan_type'] == 'b_scan']
        if not b_scan_peaks.empty:
            ax.plot(b_scan_peaks['temperature'], b_scan_peaks['peak_voltage'], 'o-', label='b_scan', color=COLOR_B_AXIS, linewidth=2, markersize=8)

        # Plot c_scan
        c_scan_peaks = df_peak_tmr[df_peak_tmr['scan_type'] == 'c_scan']
        if not c_scan_peaks.empty:
            ax.plot(c_scan_peaks['temperature'], c_scan_peaks['peak_voltage'], 's-', label='c_scan', color=COLOR_C_AXIS, linewidth=2, markersize=8)

        ax.set_xlabel('$T_K$')
        ax.set_ylabel('$V_{bias}^{max(MR)}$')
        ax.set_title('Voltage at Peak MR vs Temperature', fontweight='bold')
        ax.grid(False)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No peak TMR data found after filtering.")
else:
    print("df_combined not found in namespace.")


In [ ]:
# Plot maximum TMR for positive bias voltage and maximum TMR for negative bias voltage
if peak_data_pos :
    fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

    if peak_data_pos:
        df_pos = pd.DataFrame(peak_data_pos)
        b_scan_pos = df_pos[df_pos['scan_type'] == 'b_scan']
        c_scan_pos = df_pos[df_pos['scan_type'] == 'c_scan']
        
        if not b_scan_pos.empty:
            ax.plot(b_scan_pos['temperature'], 100*b_scan_pos['max_tmr']-100, 'o-', label='b scan', color=COLOR_B_AXIS, linewidth=2, markersize=8)
        if not c_scan_pos.empty:
            ax.plot(c_scan_pos['temperature'], 100*c_scan_pos['max_tmr']-100, 's-', label='c scan', color=COLOR_C_AXIS, linewidth=2, markersize=8)

    
    ax.set_xlabel('$T$ (K)')
    ax.set_ylabel('max($MR$ ratio) (%)')

    ax.set_xlim(15, 165)
    ax.set_ylim(0, 400)
    #ax.set_title('Maximum TMR ratio (%) (Positive and Negative Bias) vs Temperature', fontweight='bold')
    #ax.grid(False)
    
    # Place legend outside to avoid obscuring data
    ax.legend( loc='upper left')
    
    plt.tight_layout()
    plt.show()

else:
    print("Required data not found. Please run the previous cell first.")


In [ ]:
import numpy as np
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

temperatures = [100, 90, 80, 70, 60, 50, 30, 20]

def get_fn_coords(data, peak_voltage):
    """Get the FN plot coordinates (1/V, ln|I|/V²) at the peak voltage."""
    # Find the row closest to peak_voltage
    idx = (data['Voltage (V)'] - peak_voltage).abs().idxmin()
    V = data.loc[idx, 'Voltage (V)']
    I = data.loc[idx, 'I_apar (A)']
    if abs(V) >= 0.1 and I != 0:
        return 1/V, np.log(abs(I) / V**2)
    return None, None

for scan_type, ax in [('b_scan', ax1), ('c_scan', ax2)]:
    
    # --- FN curves ---
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type) &
            (df_combined['Voltage (V)'].abs() >= 0.02)
        ].copy()
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            x = 1 / data['Voltage (V)']
            y = np.log(data['I_apar (A)'].abs() / data['Voltage (V)']**2)
            ax.plot(x, y, 'o', color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

    # --- Peak TMR markers ---
    for i, temp in enumerate(temperatures):
        data_full = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy()
        if len(data_full) == 0:
            continue

        try:
            filtered_data = apply_filters(data_full, temp, i_apar_threshold, voltage_cutoffs)
        except NameError:
            filtered_data = data_full

        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]

        # Positive side peak
        data_pos = filtered_data[filtered_data['Voltage (V)'] > 0]
        if len(data_pos) > 0:
            peak_voltage_pos = data_pos.loc[data_pos['TMR_Ratio'].idxmax(), 'Voltage (V)']
            x_pos, y_pos = get_fn_coords(data_full, peak_voltage_pos)
            if x_pos is not None:
                ax.scatter(x_pos, y_pos, color="white", s=120, marker='*', 
                          edgecolors='#D55E00', linewidths=0.8, zorder=5)

        # Negative side peak
        data_neg = filtered_data[filtered_data['Voltage (V)'] < 0]
        if len(data_neg) > 0:
            peak_voltage_neg = data_neg.loc[data_neg['TMR_Ratio'].idxmax(), 'Voltage (V)']
            x_neg, y_neg = get_fn_coords(data_full, peak_voltage_neg)
            if x_neg is not None:
                ax.scatter(x_neg, y_neg, color="white", s=120, marker='*',
                          edgecolors='#D55E00', linewidths=0.8, zorder=5)

    ax.set_xlabel('$1/V_{bias}$ (V$^{-1}$)')
    ax.set_ylabel('$\ln(|I|/V_{bias}^2)$')
    ax.set_title(f'Fowler-Nordheim Plot (AFM) - {scan_type}', fontweight='bold')
    ax.grid(False)
    ax.legend()

# Add a legend entry for the peak markers
from matplotlib.lines import Line2D
marker_legend = Line2D([0], [0], marker='*', color='w', markerfacecolor='white',
                        markeredgecolor=COLOR_C_AXIS, markersize=12, label='Peak MR voltage')
ax1.legend(handles=ax1.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax1.get_legend_handles_labels()[1] + ['Peak MR voltage'])
ax2.legend(handles=ax2.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax2.get_legend_handles_labels()[1] + ['Peak MR voltage'])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

temperatures = [100, 90, 80, 70, 60, 50, 30, 20]

def get_fn_coords_par(data, peak_voltage):
    """Get the FN plot coordinates (1/V, ln|I_par|/V²) at the peak voltage."""
    idx = (data['Voltage (V)'] - peak_voltage).abs().idxmin()
    V = data.loc[idx, 'Voltage (V)']
    I = data.loc[idx, 'I_par (A)']
    if abs(V) >= 0.1 and I != 0:
        return 1/V, np.log(abs(I) / V**2)
    return None, None

for scan_type, ax in [('b_scan', ax1), ('c_scan', ax2)]:

    # --- FN curves ---
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type) &
            (df_combined['Voltage (V)'].abs() >= 0.02)
        ].copy()
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            x = 1 / data['Voltage (V)']
            y = np.log(data['I_par (A)'].abs() / data['Voltage (V)']**2)
            ax.plot(x, y, 'o', color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

    # --- Peak TMR markers ---
    for i, temp in enumerate(temperatures):
        data_full = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy()
        if len(data_full) == 0:
            continue

        try:
            filtered_data = apply_filters(data_full, temp, i_apar_threshold, voltage_cutoffs)
        except NameError:
            filtered_data = data_full

        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]

        # Positive side peak
        data_pos = filtered_data[filtered_data['Voltage (V)'] > 0]
        if len(data_pos) > 0:
            peak_voltage_pos = data_pos.loc[data_pos['TMR_Ratio'].idxmax(), 'Voltage (V)']
            x_pos, y_pos = get_fn_coords_par(data_full, peak_voltage_pos)
            if x_pos is not None:
                ax.scatter(x_pos, y_pos, color='white', s=120, marker='*',
                          edgecolors='#D55E00', linewidths=0.8, zorder=5)

        # Negative side peak
        data_neg = filtered_data[filtered_data['Voltage (V)'] < 0]
        if len(data_neg) > 0:
            peak_voltage_neg = data_neg.loc[data_neg['TMR_Ratio'].idxmax(), 'Voltage (V)']
            x_neg, y_neg = get_fn_coords_par(data_full, peak_voltage_neg)
            if x_neg is not None:
                ax.scatter(x_neg, y_neg, color='white', s=120, marker='*',
                          edgecolors='#D55E00', linewidths=0.8, zorder=5)

    ax.set_xlabel('$1/V_{bias}$ (V$^{-1}$)')
    ax.set_ylabel('$\ln(|I_{FM}|/V_{bias}^2)$')
    ax.set_title(f'Fowler-Nordheim Plot (FM) - {scan_type}', fontweight='bold')
    ax.grid(False)
    ax.legend()

# Add legend entry for peak markers
from matplotlib.lines import Line2D
marker_legend = Line2D([0], [0], marker='*', color='w', markerfacecolor='white',
                        markeredgecolor=COLOR_C_AXIS, markersize=12, label='Peak MR voltage')
ax1.legend(handles=ax1.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax1.get_legend_handles_labels()[1] + ['Peak MR voltage'])
ax2.legend(handles=ax2.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax2.get_legend_handles_labels()[1] + ['Peak MR voltage'])

plt.tight_layout()
plt.show()

In [ ]:
# Plot maximum TMR for positive bias voltage and maximum TMR for negative bias voltage
if peak_data_pos :
    fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

    if peak_data_pos:
        df_pos = pd.DataFrame(peak_data_pos)
        b_scan_pos = df_pos[df_pos['scan_type'] == 'b_scan']
        c_scan_pos = df_pos[df_pos['scan_type'] == 'c_scan']
        
        if not b_scan_pos.empty:
            ax.plot(b_scan_pos['temperature'], 100*b_scan_pos['max_tmr']-100, 'o-', label='$b$ scan', color=COLOR_B_AXIS, linewidth=2, markersize=8)
        if not c_scan_pos.empty:
            ax.plot(c_scan_pos['temperature'], 100*c_scan_pos['max_tmr']-100, 's-', label='$c$ scan', color=COLOR_C_AXIS, linewidth=2, markersize=8)

    
    ax.set_xlabel('$T$ (K)')
    ax.set_ylabel('max($MR$ ratio) (%)')

    ax.set_xlim(15, 165)
    ax.set_ylim(0, 400)
    #ax.set_title('Maximum TMR ratio (%) (Positive and Negative Bias) vs Temperature', fontweight='bold')
    #ax.grid(False)
    
    # Place legend outside to avoid obscuring data
    ax.legend( loc='upper left')
    
    plt.tight_layout()
    plt.show()

else:
    print("Required data not found. Please run the previous cell first.")
